# Fine-tune Qwen3-Embedding for code embeddings using Amazon SageMaker
## Introduction
Embedding models for code have become essential components in modern software development workflows, powering applications like semantic code search, retrieval-augmented generation (RAG), and intelligent coding assistants. These models capture the semantic and functional relationships between code snippets by transforming them into numerical vector representations, enabling more effective code retrieval and understanding compared to traditional methods that treat code as sequences of characters.

While general-purpose embedding models perform well on natural language tasks, code has unique characteristics such as: syntax, structure, and domain-specific semantics - that benefit from specialized fine-tuning. This blog demonstrates how to fine-tune the latest [Qwen3-Embedding](https://github.com/QwenLM/Qwen3-Embedding) models specifically for code embeddings using Amazon SageMaker, showcasing three key innovations that make this approach particularly effective: (1) leveraging the state-of-the-art Qwen3-Embedding foundation model, (2) implementing Matryoshka Representation Learning to flexibly handle different dimensionality requirements, and (3) applying Contrastive Learning specifically optimized for code relationships.

First, we'll explore the Qwen3-Embedding model family, representing the latest advancement in embedding technology with state-of-the-art performance on code benchmarks and built-in instruction awareness. Next, we'll implement Matryoshka Representation Learning, an innovative technique that creates flexible, nested embeddings allowing you to balance performance with computational efficiency by truncating vectors to optimal dimensions. Finally, we'll leverage contrastive learning, a powerful training paradigm that teaches models to understand code semantics by learning from naturally occurring code-documentation pairs found in real-world repositories.

Together, these three approaches create a comprehensive solution for building high-performance code embedding models that can adapt to various deployment scenarios while maintaining superior accuracy on code understanding tasks.
## Introducing Qwen3-Embedding: The Next-Generation Model Family for Code Embeddings
The Qwen3-Embedding series represents one of the latest advancement in Alibaba's Qwen model family, specifically engineered for text embedding, retrieval, and ranking tasks with exceptional performance in code understanding applications. Built upon the robust foundational architecture of the Qwen3 series, these models inherit exceptional multilingual capabilities, long-text understanding, and advanced reasoning skills that make them particularly well-suited for fine-tuning on code embedding tasks. The series offers comprehensive flexibility through three distinct model sizes-0.6B, 4B, and 8B parameters-allowing developers to choose the optimal balance between computational efficiency and performance for their specific deployment requirements .

### Key Benefits of Qwen3-Embedding model for code embedding fine-tuning
__Model Size Options__  
Qwen3-Embedding provides three model configurations (0.6B, 4B, and 8B parameters) that cater to diverse computational constraints and performance requirements. This flexibility enables selecting the most appropriate model size based on infrastructure limitations, latency requirements, and accuracy targets, with even the smallest 0.6B model demonstrating competitive performance on code retrieval benchmarks.

__Built-in Instruction Awareness__  
All Qwen3-Embedding models feature native instruction awareness, supporting user-defined instructions to enhance performance for specific tasks, languages, or scenarios without requiring separate "instruct" variants. Qwen3-Embedding evaluation indicates that utilizing instructions typically yields performance improvements of 1% to 5% compared to non-instructed approaches, making this capability invaluable for optimizing code embedding tasks.

__Extended Context Length__  
The series supports an impressive 32K token context length across all model sizes, significantly exceeding many competing embedding models and enabling comprehensive processing of large code files, extensive documentation, and complex multi-file codebases. This extended context capability is particularly beneficial for code embedding applications where understanding broader code structure and relationships is crucial for generating meaningful representations.

__Superior Code Retrieval Capabilities__  
Qwen3-Embedding models have achieved state-of-the-art performance on code-specific benchmarks, with the flagship 8B model scoring high on the [MTEB](https://huggingface.co/spaces/mteb/leaderboard) Code benchmark. The models excel across multiple code-related applications including code retrieval, code classification, and code clustering, demonstrating their versatility for various code embedding use cases .

__Multilingual Programming Language Support__  
The Qwen3-Embedding series offers robust support for over 100 languages, including comprehensive coverage of major programming languages such as Python, Java, JavaScript, C++, SQL, and many others. This extensive programming language support, inherited from the multilingual capabilities of the Qwen3 foundation models, enables effective cross-language code understanding and retrieval, making the models ideal for diverse development environments and polyglot codebases. The models provide strong multilingual, cross-lingual, and code retrieval capabilities that are essential for modern software development workflows involving multiple programming languages and frameworks.
## Matryoshka Representation Learning: Flexible Embedding Dimensions with Qwen3 Models
Traditional embedding models face a fundamental limitation: they generate fixed-dimensional vectors where all dimensions carry equal importance, requiring organizations to deploy separate models for different computational requirements. For instance, mobile applications might need lightweight 256-dimensional embeddings for real-time processing, while server-side analytics could benefit from rich 1024-dimensional representations for maximum accuracy.

[Matryoshka Representation Learning](https://arxiv.org/abs/2205.13147) (MRL) addresses this challenge by training embedding models to prioritize information hierarchically, with the most critical semantic content concentrated in the earlier dimensions. This approach, inspired by Russian nesting dolls, enables a single model to produce high-quality embeddings at multiple dimensional scales. For example, when processing the code snippet ```def quicksort(arr): return sorted(arr)```, with an MRL-trained model:
- First 256 dimensions: Contain 80% of the important information (core semantic meaning)
- First 512 dimensions: Contain 90% of the important information (additional contextual information)
- All 1024 dimensions: Contain 100% of the information (fine-grained details)

The key advantage is deployment flexibility: organizations can deploy one model that serves multiple use cases by specifying the desired output dimension at inference time. A mobile app can request 256-dimensional embeddings for fast similarity searches, while a code analysis system can request 1024-dimensional embeddings for comprehensive semantic understanding-all from the same deployed model. This "one model fits all" approach significantly reduces infrastructure complexity, maintenance overhead, and deployment costs while providing the flexibility to optimize performance based on real-time computational constraints.

__Motivation for using Matryoshka with Qwen3 embedding models__  
The primary motivation for Matryoshka embeddings stems from the practical challenges of deploying high-dimensional embedding models in resource-constrained environments. Consider the Qwen3-Embedding model family we're working with: the smallest Qwen3-Embedding-0.6B model produces embeddings with up to 1,024 dimensions, the Qwen3-Embedding-4B model generates vectors up to 2,560 dimensions, and the flagship Qwen3-Embedding-8B model creates massive 4,096-dimensional vectors. These substantial vector sizes consume significant memory and computational resources - for instance, storing just one million 4,096-dimensional embeddings requires approximately 16GB of memory in float32 precision, making them expensive to deploy and slow to process at scale.

Matryoshka addresses this challenge by training the Qwen3 models to frontload the most important semantic information into the first few dimensions, creating a hierarchy where earlier dimensions capture broad semantic meaning while later dimensions add progressively finer details. This design philosophy allows developers to choose the appropriate level of detail needed for specific tasks - you could truncate the 4,096-dimensional output from Qwen3-Embedding-8B down to 512 dimensions by passing a dimension value parameter during inference request.

__Key Benefits for Qwen3 Embedding Deployment__  
Matryoshka embeddings offer several compelling advantages that make them particularly valuable for production applications using Qwen3 models. Storage and Speed Efficiency: For some use-cases, the technique can achieve [up to 14x smaller embedding sizes](https://arxiv.org/abs/2205.13147) while maintaining comparable accuracy, resulting in significant cost savings for storage and faster retrieval operations - particularly important when dealing with Qwen3's large dimensional outputs ranging from 1,024 to 4,096 dimensions. Flexible Performance Trade-offs: Organizations can scale their Qwen3 embedding solutions to match desired storage costs, processing speeds, and performance requirements by simply truncating embeddings to appropriate dimensions - for example, reducing Qwen3-Embedding-8B's 4,096 dimensions to 512 for memory-constrained environments.

Deployment Simplicity: Unlike traditional dimensionality reduction techniques that require retraining or additional processing steps, Matryoshka optimization is built into the Qwen3 training process, eliminating extra pipeline complexity. Robust Performance Retention: Research demonstrates that Matryoshka models can preserve up to 98.37% of their original performance even when truncated to just 8.3% of the original embedding size ([ref](https://huggingface.co/blog/matryoshka)). This means you could potentially reduce Qwen3-Embedding-8B's 4,096 dimensions down to approximately 340 dimensions while retaining nearly all semantic quality - making them remarkably efficient for applications requiring real-time processing of large code corpora where the full dimensional output would be prohibitively expensive to deploy and manage.
## Contrastive Learning for Code Embeddings
Contrastive learning has emerged as a powerful training paradigm for embedding models, particularly effective for code representation learning. The core principle involves training the model to pull semantically similar code snippets closer together in the embedding space while pushing dissimilar ones apart.

During training, the model learns from carefully constructed pairs: positive examples include functionally equivalent code written in different styles, refactored versions of the same function, or code snippets that solve the same problem using different approaches. Negative examples consist of code with different functionality or purpose. This approach is especially valuable for code embeddings because it helps the model distinguish between superficial syntactic differences and deeper semantic similarities-for instance, learning that ```for i in range(len(list))``` and ```for item in list``` represent similar iteration patterns despite different syntax.

By optimizing contrastive objectives like [InfoNCE loss](https://paperswithcode.com/method/infonce) or triplet loss, the fine-tuned Qwen3-embedding model develops a more nuanced understanding of code semantics, resulting in embeddings that better capture functional relationships and enable more accurate code search, similarity detection, and retrieval-augmented generation tasks.

__How Contrastive Learning Works in Practice__  
The contrastive learning framework trains models through structured comparison of code pairs within each training batch. Rather than learning from isolated examples, the model simultaneously processes multiple code-description pairs, learning to associate functionally equivalent implementations while differentiating them from unrelated code snippets.

For example, the system learns to connect various sorting algorithm implementations-bubble sort, quicksort, merge sort-with descriptions like "arranges array elements in ascending order" while distinguishing these from functionally different operations like database queries or string manipulations. This approach leverages techniques such as Multiple Negatives Ranking Loss, where each positive code-description pair in a training batch serves as a negative example for all other pairs in the same batch. This creates dense training signals that efficiently teach the model to recognize functional similarity across different syntactic representations.


# Let's build
## Start with few imports

In [ ]:
import sagemaker
import boto3
import os
from datetime import datetime
from sagemaker.huggingface import HuggingFace, HuggingFaceModel
from sagemaker.s3 import S3Uploader


## Configure the SageMaker environment 
Configure IAM role for SageMaker so it can interract with services such as: EC2, S3, etc.
I use SageMaker default S3 bucket which is automatically created if you don't specify one when using SageMaker's Python SDK.

In [ ]:
sess = sagemaker.Session()

# S3 bucket information
s3_bucket = sess.default_bucket()
prefix = "qwen3-embedding-code"
s3_path = f"s3://{s3_bucket}/{prefix}"

# Role for SageMaker to access S3, EC2 and other services
try:
	role = sagemaker.get_execution_role()
except ValueError:
	iam = boto3.client('iam')
	role = iam.get_role(RoleName='sagemaker_execution_role')['Role']['Arn']
    
print(f"SageMaker role ARN: {role}")
print(f"SageMaker session bucket: {s3_path}")
print(f"AWS region: {sess.boto_region_name}")


## Create a training dataset
We'll create a training dataset specifically designed for code embedding tasks using code-description pairs that reflect real-world development scenarios.
The script that generate the code samples can be found at `scripts/dataset.py`.
Generated dataset is transformed to `jsonl` format and uploaded to S3, later to be used by the training job.

In [ ]:
# Create and display dataset
from scripts.dataset import create_code_embedding_dataset

train_dataset = create_code_embedding_dataset()
print(f"Created training dataset with {len(train_dataset)} code-description pairs")
print("Sample entries:")
for i in range(5):
    print(f"  Code: {train_dataset[i]['text1'][:60]}...")
    print(f"  Description: {train_dataset[i]['text2']}")
    print()


Prepare and upload the training dataset to S3 for SageMaker access

In [ ]:
# Save dataset locally
train_dataset.to_json('train_dataset.jsonl')

# Upload training data to S3
training_input_path = S3Uploader.upload(
    local_path='train_dataset.jsonl',
    desired_s3_uri=f'{s3_path}/train'
)

os.remove('train_dataset.jsonl')
print(f"Training data uploaded to: {training_input_path}")

## Training configurations
Set up the HuggingFace estimator with appropriate configurations for Qwen3-Embedding fine-tuning.  
We use `requirements.txt` to overide the pre-built [HaggingFace deep learning container](https://docs.aws.amazon.com/sagemaker/latest/dg/hugging-face.html) (DLC) with the package versions we need. This is because current HuggingFace DLC doesn't sutissfy `transformers>=4.51.0`. Overiding this pre-built using `requirements.txt` help us to avoid writing [custom container](https://docs.aws.amazon.com/sagemaker/latest/dg/docker-containers-adapt-your-own.html).  

The target_dimension hyperparameter serves to guide Matryoshka Loss on which embedding size to prioritize during multi-dimensional training. Although the base Qwen3-Embedding model produces full-length vectors (e.g., 1,024 dimensions for the 0.6B variant), specifying target_dimension=512 during training helps the model front-load its most critical semantic information into the first 512 dimensions. This ensures that, when truncated at inference time, the 512-dimensional slice retains maximal performance without requiring a separate projection step.
- Without this guidance, the model would distribute key features uniformly across all 1,024 dimensions, making any fixed truncation suboptimal.
- Training with a target_dimension directs Matryoshka Loss to weight those dimensions more heavily, improving downstream retrieval accuracy at 512 dimensions.

__Note__: Training will take appox. 8 minutes.
### SageMaker Training Managed Warm Pools
When you initiate a SageMaker training job, on every job startup, the service pulls the specified Docker image (often few GB of size) before provisioning the training instances. This is followed by downloading the model  which can also be few GB in size. This alone can add tens of seconds to minutes of overhead per run. SageMaker provides [AI Managed Warm Pools](https://docs.aws.amazon.com/sagemaker/latest/dg/train-warm-pools.html) to keep infrastructure-including the container image=“warm” between jobs, significantly reducing startup latency.  

__Warm Pools__ allow you to retain provisioned resources for a configurable period after a training job finishes. When a new training job with matching configuration parameters starts within the warm pool’s lifetime, SageMaker reuses the existing instances-and crucially, the already-pulled container image-avoiding repeated downloads.
You can opt in to warm pools through the SageMaker Python SDK by specifying `keep_alive_period_in_seconds` when creating your estimator.  

When you enable SageMaker AI Managed Warm Pools, the underlying EC2 instances and the pulled container image remain “warm” for the specified `keep_alive_period_in_seconds`, but your training job’s configuration-including hyperparameters and your training script-always comes from the new job request you submit.  
You can further read [Best practices for Amazon SageMaker Training Managed Warm Pools](https://aws.amazon.com/blogs/machine-learning/best-practices-for-amazon-sagemaker-training-managed-warm-pools/) blog.

__Note__: If a training job is created with `keep_alive_period_in_seconds` specified, but you did not request a warm pool limit increase, then a warm pool is not retained after the completion of the training job. A warm pool is only created if your warm pool limit has sufficient resources.

In [ ]:
# Configure hyperparameters for Qwen3-Embedding-0.6B fine-tuning. Will pass to train.py
hyperparameters = {
    'model_name': 'Qwen/Qwen3-Embedding-0.6B',
    'num_train_epochs': 3,
    'per_device_train_batch_size': 12,
    'learning_rate': 2e-5,
    'warmup_ratio': 0.1,
    'target_dimension': 512,
    'logging_steps': 100,
    'fp16': True,
    "save_strategy": "no",  # disables saving checkpoints to reduce model packaging size (not for prod)
}

# Create HuggingFace Estimator
# Note:
#  - Qwen3 models require transformers>=4.51.0; Below code will result container with transformers_version='4.49.0'
#  - The requirements.txt under training_code directory will be used by SageMaker to upgrade the framework to the required version
huggingface_estimator = HuggingFace(
    entry_point='train.py',
    source_dir='training_code',           # Directory with train.py & requirements.txt
    instance_type='ml.g4dn.xlarge',
    instance_count=1,
    role=role,
    transformers_version='4.49.0',        # Upgraded via requirements.txt
    pytorch_version='2.5.1',
    py_version='py311',
    hyperparameters=hyperparameters,
    base_job_name='qwen3-embed-finetune', # Prefix for job naming
    volume_size=100,
    keep_alive_period_in_seconds=1800,   # Retain provisioned resources for 30 minutes
)

In [ ]:
# Start the fine-tuning job
print("Starting Qwen3-Embedding fine-tuning job...")
print(f"Timestamp: {datetime.now()}")

# Start the training job with the jsonl dataset location in S3
huggingface_estimator.fit({
    'train': training_input_path
})

training_job_name = huggingface_estimator.latest_training_job.name
model_data = huggingface_estimator.model_data
print(f"Model artifacts location: {model_data}")
print(f"\nTraining job launched: {training_job_name}")

## That's it
Our fine-tuned model is waiting for us in S3
## Let's deploy the model and put it to the test
Deploying a model with SageMaker's HuggingFaceModel involves a multi-stage process that begins on your local environment and completes in the AWS cloud. When you call the `.deploy()` method, SageMaker initiates a model repacking operation that downloads your model artifacts from S3, combines them with your custom inference code (`inference.py` and other files in the `inference_code` directory), and creates a consolidated model package. This repacking process occurs on your local environment - the notebook instance's root filesystem (overlay mount) in our case. This step requires sufficient disk space—typically 2-3 times the model size. After repacking, SageMaker uploads the combined artifact to a new S3 location and provisions the specified compute resources (`ml.g4dn.xlarge` in this example).  
Once the infrastructure is ready, SageMaker pulls the appropriate Deep Learning Container (DLC) image based on your framework specifications (`transformers_version='4.49.0'`, `pytorch_version='2.6.0'`, `py_version='py312'`), downloads the repacked model artifact from S3, and extracts it to the container's file system. The container then initializes the model server, which loads your model into memory and executes your inference handler code specified in `entry_point='inference.py'`. This handler implements standardized functions like `model_fn()`, `input_fn()`, `predict_fn()`, and `output_fn()` that process incoming requests and return predictions. After successful initialization, SageMaker exposes the endpoint for real-time inference requests through a REST API, allowing you to invoke predictions using the returned predictor object.

__Note__: It might take long time for the model repacking and endpoint deployment. Be patient.

In [ ]:
 # Create HuggingFace model for deployment
huggingface_model = HuggingFaceModel(
    model_data=model_data,
    role=role,
    entry_point='inference.py',
    source_dir='./inference_code',
    transformers_version='4.49.0',
    pytorch_version='2.6.0',
    py_version='py312'
)

In [ ]:
# Deploy the model
predictor = huggingface_model.deploy(
    initial_instance_count=1,
    instance_type='ml.g4dn.xlarge',
    endpoint_name="qwen3-embedding-code-finetune"
)

## Predictions
Creating a Predictor class from the deployed model endpoint name. This helps us to create a predictor without neededing to run the deployment API again. Use the deployed endpoint name to create a predictor.  
The prediction supports two operations on the same endpoint:
* encode - Outputs an embedding for the given input and dimension
* similarity - Outputs similarity score for the given text1 & text2 along their embeddings

In [ ]:
import sagemaker
from sagemaker.deserializers import JSONDeserializer
from sagemaker.serializers import JSONSerializer

predictor = sagemaker.Predictor(
    endpoint_name="qwen3-embedding-code-finetune",
    sagemaker_session=sagemaker.Session(),
    serializer=JSONSerializer(),
    deserializer=JSONDeserializer()
)

### Similarity tests with dynamic embedding dimensions
By adjusting the dimension parameter, you can see how truncating the same underlying embedding model at 1024, 728, 512, 256, or 128 dimensions influences both the cosine similarity score and the returned embedding length. This dynamic resizing is a direct benefit of the Matryoshka training approach, which frontloads semantic information into the earliest vector dimensions. We encourage you to run these examples, observe how similarity varies, and then craft your own code pairs-experimenting with related and unrelated snippets to deepen your intuition about embedding truncation and performance.

In [ ]:
# Encode a Python function at 128 dimensions
predictor.predict({
    "operation": "encode",
    "inputs": ["def quicksort(arr): return sorted(arr)"],
    "dimension": 128
})

In [ ]:
# Perfect match at minimal dimension
predictor.predict({
    "operation": "similarity",
    "text1": "def add(a, b): return a + b",
    "text2": "def add(a, b): return a + b",
    "dimension": 128
})

In [ ]:
# Dissimilar snippets comparison
predictor.predict({
    "operation": "similarity",
    "text1": "print('Hello World')",
    "text2": "SELECT * FROM users",
    "dimension": 512
})

In [ ]:
# Compute similarity for the same functionality, but different programming languages
# Test the effect of non-equivalent function names
factorial_py = "def factorial(n): return 1 if n <= 1 else n * factorial(n-1)"
factorial_java = "public static int factorial(int n) {\n    if (n <= 1) return 1;\n    return n * factorial(n - 1);\n}"
predictor.predict({
    "operation": "similarity",
    "text1": factorial_py,
    "text2": factorial_java,
    "dimension": 128
})